# 《PythAPCS123》單元 13-3：執行時期錯誤（Runtime Error, RE）常見排行榜與崩潰防禦

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnyy-lab/APCS1to3/blob/main/PythAPCS123_13-3_runtime_errors_ranking_and_defense.ipynb)

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者
**核心目標**：徹底告別上傳 Online Judge 卻莫名拿到 Runtime Error（RE）的恐懼！深入透視 Python 程式在執行中途突發例外（Exception）而強制中斷的底層機制，學會由下而上閱讀多層呼叫堆疊（Call Stack）。地毯式剖析 APCS 考場最常出現的五大 RE 殺手：`IndexError`（含一維越界、二維網格負數索引陷阱、空佇列 pop）、`ValueError`（浮點字串轉整數、解包數量不符、index 找不到）、`KeyError`（計數器未初始化、set.remove 崩潰）、`ZeroDivisionError`（除以零與模除零）與 `TypeError`（字串加整數、把串列當中括號呼叫）。掌握「走訪前先檢查（Look Before You Leap, LBYL）」的邊界防禦哲學，築起銅牆鐵壁般的條件式護城河！


### 13.3.1 什麼是例外（Exception）？程式執行中途突發崩潰的觸發機制

在前一節中我們學習到，語法錯誤（SyntaxError）發生在直譯器的「編譯剖析期」，只要語法不合規格，整份程式連半個字元都不會執行。而本節要探討的**「執行時期錯誤（Runtime Error，在 OJ 評判系統簡稱為 RE）」**，其性質則截然不同：你的程式碼在文法結構上是完全合法且符合 Python 語法標準的，因此直譯器能夠順利啟動並開始一行行執行。

然而，當程式跑到某一條指令時，突然遭遇了電腦「在邏輯上或物理上無法繼續處理的突發狀況」——例如要求從一個空箱子裡拿東西、要求計算除以零、或是要求把英文字母 `"abc"` 強制轉成十進位整數。此時 Python 直譯器無法憑空猜測該如何往下執行，它唯一的自保手段就是拋出一個**「例外（Exception 物件）」**。如果程式設計師沒有預先寫好攔截防禦措施，直譯器就會當場強制終止程式，並在終端機噴出長串的紅色 Traceback 報錯。在 APCS 考場上，只要有一筆隱藏測資引發未捕捉的例外，該測試點就會直接被判定為 RE，拿到 0 分！

**如何閱讀多層函式呼叫引爆的 Traceback？**
在真實題目中，錯誤往往發生在多層函式呼叫的深處。閱讀 Traceback 的黃金法則為**「由下而上（Bottom-Up）」**：
1. **看最底層**：確認引爆崩潰的具體例外型別（例如 `IndexError`）與文字描述。
2. **看倒數第二行**：找到真正爆炸的那一行具體指令（真正引發例外的最深層程式碼）。
3. **往上看調用鏈**：檢查最上層的主程式傳入了什麼樣的極端參數，導致底層函式消化不良而暴斃！


In [ ]:
# 13.3.1 程式碼演示：多層呼叫堆疊（Call Stack）崩潰現場透視與「由下而上」閱讀法
import traceback

def calculate_discount(price, rate):
    # 最底層計算函式：若 rate 為負數或除以零
    if rate == 0:
        raise ZeroDivisionError("折扣率不可為 0！")
    return price / rate

def process_single_order(order_id, item_price, discount_rate):
    # 中間業務處理層
    print(f"  --> 正在處理訂單 #{order_id}，商品價格={item_price}...")
    final_price = calculate_discount(item_price, discount_rate)
    return final_price

def main_system():
    # 頂層主程式
    orders = [(101, 500, 2), (102, 800, 0)] # 訂單 102 包含危險的折扣率 0
    print("=== 訂單系統啟動 ===")
    for oid, price, rate in orders:
        print(f"\n[主系統] 派發訂單 {oid}:")
        process_single_order(oid, price, rate)

print("--- 模擬執行多層呼叫系統（觀察 Traceback 層級）---")
try:
    main_system()
except Exception as e:
    print("\n" + "="*50)
    print("🚨 [Traceback 呼叫堆疊解剖室] 🚨")
    traceback.print_exc(limit=3)
    print("="*50)
    print("由下而上解讀秘笈：")
    print("1. [最底層] calculate_discount 拋出 ZeroDivisionError (事故第一現場)")
    print("2. [中間層] process_single_order 第 13 行呼叫 calculate_discount")
    print("3. [頂層] main_system 第 22 行傳入了 rate=0 觸發了這一連串雪崩！")


### 13.3.1 語法重點回顧與核心觀念提煉

追蹤執行時期崩潰的四大關鍵心智模型：
1. **多層呼叫由下而上看**：底層告訴你「怎麼死的」，上層告訴你「誰把毒藥送進來的」。
2. **前期正常運作**：程式在第 1 筆訂單完全正常，直到第 2 筆訂單才暴斃。這說明本機測試若只測了 1 筆常規資料，根本無法發現潛伏的例外地雷！
3. **例外物件化**：Python 中每個例外都是繼承自 `BaseException` 的物件，攜帶完整的型態資訊與錯誤字串。
4. **防禦優先原則**：與其讓程式跑到最深層崩潰，不如在主程式入口處就先把不合法的參數攔截下來！


In [ ]:
# 13.3.1 學生實作練習：安全除法防禦器
# 任務說明：實作 safe_divide_orders(total_amount, num_people) 函式
# 計算每個人應分攤的金額（total_amount / num_people）
# 邊界防禦要求：
# 1. 若 num_people <= 0，不可進行除法，應安全回傳 None 而非引爆 ZeroDivisionError！
# 2. 若參數合法，回傳計算出的每人分攤金額（float）

def safe_divide_orders(total_amount: float, num_people: int):
    # 請在此處進行分母防禦
    if num_people <= 0:
        return None
    return total_amount / num_people

# 測試用例
print("正常平分 1000 元給 4 人:", safe_divide_orders(1000, 4))
print("防禦平分給 0 人:", safe_divide_orders(1000, 0))
print("防禦平分給 -2 人:", safe_divide_orders(1000, -2))


In [ ]:
# 13.3.1 單元測試驗證
assert safe_divide_orders(100, 4) == 25.0
assert safe_divide_orders(50, 2) == 25.0
assert safe_divide_orders(100, 0) is None
assert safe_divide_orders(100, -5) is None
print("🎉 13.3.1 所有測試通過！成功建立由下而上排查與分母防禦思維！")


### 13.3.2 RE 排行榜榜首：IndexError: list index out of range

在 Online Judge 與 APCS 實作題所有送出紀錄中，`IndexError` 長年穩居各類崩潰排行榜的「冠軍寶座」。此錯誤的官方訊息通常為 `IndexError: list index out of range`（串列索引超出範圍），意指你試圖用一個不存在的索引號碼去存取串列元素。

考場四大經典 `IndexError` 引爆場景：
1. **0-based 索引轉換盲點**：串列長度為 $N$ 時，合法的正向索引僅有 $0$ 到 $N-1$。初學者常誤用 `arr[N]` 來試圖讀取「最後一個元素」，直接導致越界崩潰（正確寫法應為 `arr[N-1]` 或 `arr[-1]`）。
2. **走訪相鄰比對差一（Off-by-one）**：在檢查相鄰元素 `arr[i] == arr[i+1]` 時，若外層寫 `range(len(arr))`，最後一輪必然越界；正確上限必須嚴格減一：`range(len(arr) - 1)`。
3. **二維網格相鄰探測與「負數索引幽靈」**：
   在網格尋路題目中，探測四方向鄰居 $(r-1, c), (r+1, c), (r, c-1), (r, c+1)$。初學者最常忽略：**在 Python 中，`grid[-1][c]` 不會報錯，而是會自動繞到最後一列（負數索引）**！這會引爆極度可怕的「幽靈穿牆」邏輯錯誤；而探測超出下界 `grid[H][c]` 則會引爆 `IndexError`。因此，嚴格的雙向邊界檢查 `0 <= nr < H and 0 <= nc < W` 是唯一正解！
4. **空容器彈出（Pop from empty list）**：
   在模擬佇列（Queue）或堆疊（Stack）時，使用 `arr.pop(0)` 或 `arr.pop()`，一旦串列已被抽空卻未檢查 `if arr:` 就再次執行 pop，直譯器會立刻拋出 `IndexError: pop from empty list`！


In [ ]:
# 13.3.2 程式碼演示：IndexError 四大高頻場景（相鄰、網格、空佇列 pop）
print("--- 場景 1: 相鄰比對越界 vs 安全邊界 ---")
data = [10, 20, 20, 30]
try:
    # 錯誤寫法：range(len(data)) 走到最後一輪 i=3 時，data[4] 越界！
    for i in range(len(data)):
        if data[i] == data[i + 1]:
            pass
except IndexError as e:
    print(f"  [捕捉錯誤 1] IndexError: {e} (相鄰比對忘記 len - 1)")

print("\n--- 場景 2: 二維網格邊界探測與負數幽靈穿牆 ---")
grid = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
]
H, W = 3, 3

def safe_get_cell(r, c):
    # 嚴格防禦：同時阻絕小於 0（防止穿牆）與大於等於 H, W（防止越界）
    if 0 <= r < H and 0 <= c < W:
        return grid[r][c]
    return None # 邊界外出界

print("探測合法座標 (1, 1):", safe_get_cell(1, 1))
print("探測上邊界越界 (-1, 0):", safe_get_cell(-1, 0), "(成功阻絕 Python 負數倒數取值幽靈！)")
print("探測下邊界越界 (3, 0):", safe_get_cell(3, 0), "(成功防禦 IndexError！)")

print("\n--- 場景 3: 空串列 pop 崩潰與先驗防禦 ---")
queue = ["任務A"]
# 處理完第一項任務
item1 = queue.pop(0)
print(f"取出: {item1}, 佇列現狀: {queue}")

try:
    # 佇列已空，強行 pop 引爆崩潰
    item2 = queue.pop(0)
except IndexError as e:
    print(f"  [捕捉錯誤 3] IndexError: {e}")

# 正確防禦寫法：pop 前先檢查容器是否非空
if queue:
    item2 = queue.pop(0)
else:
    print("  [安全防禦] 佇列已空，停止提取，避免 pop from empty list 崩潰！")


### 13.3.2 語法重點回顧與核心觀念提煉

防範 `IndexError` 的四大黃金守則：
1. **相鄰比對必減一**：`for i in range(len(arr) - 1):` 永遠是相鄰比對的唯一合法上限。
2. **網格邊界閉區間檢查**：`0 <= r < H and 0 <= c < W`，千萬不能只寫 `r < H`，因為小於 0 會引發 Python 特有的負數繞圈（Wrap-around）邏輯臭蟲！
3. **取出前必確認非空**：呼叫 `pop()` 或存取 `arr[0]` 前，務必加上 `if arr:` 衛語句。
4. **切片寬容保護**：`arr[a:b]` 就算範圍全部超出長度，也只會平靜回傳空串列 `[]`，絕不引爆 IndexError。


In [ ]:
# 13.3.2 學生實作練習：安全網格四方向鄰居探測器
# 任務說明：實作 get_valid_neighbors(grid, r, c) 函式
# 給定二維矩陣 grid（H 列、W 欄）與當前座標 (r, c)
# 探測其上下左右四個相鄰格子的數值：(r-1, c), (r+1, c), (r, c-1), (r, c+1)
# 嚴格要求：
# 1. 嚴禁負數座標繞圈（不可存取負數索引）
# 2. 嚴禁超出網格邊界
# 3. 將所有合法相鄰位置的數值收集為串列回傳！

def get_valid_neighbors(grid: list, r: int, c: int) -> list:
    H = len(grid)
    W = len(grid[0]) if H > 0 else 0
    neighbors = []
    
    # 四方向向量：上、下、左、右
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    for dr, dc in directions:
        nr, nc = r + dr, c + dc
        # 請在此處填入嚴密的雙向邊界條件判斷
        if 0 <= nr < H and 0 <= nc < W:
            neighbors.append(grid[nr][nc])
            
    return neighbors

# 測試用例
my_grid = [
    [10, 20, 30],
    [40, 50, 60],
    [70, 80, 90]
]
print("左上角 (0, 0) 的鄰居 (只有下與右):", get_valid_neighbors(my_grid, 0, 0))
print("正中央 (1, 1) 的鄰居 (上下左右共 4 個):", get_valid_neighbors(my_grid, 1, 1))


In [ ]:
# 13.3.2 單元測試驗證
g = [
    [1, 2],
    [3, 4]
]
assert sorted(get_valid_neighbors(g, 0, 0)) == [2, 3]
assert sorted(get_valid_neighbors(g, 1, 1)) == [2, 4]
assert sorted(get_valid_neighbors(g, 0, 1)) == [1, 4]
assert get_valid_neighbors([[]], 0, 0) == []
print("🎉 13.3.2 所有測試通過！徹底征服 IndexError 與二維網格幽靈穿牆陷阱！")


### 13.3.3 型態轉換與查找失敗：ValueError

在 APCS 考場的第二大 RE 殺手是 `ValueError`。直譯器噴出此錯誤時，代表「函式接收到的引數型態是正確的，但引數的值（Value）內容不符合該函式的處理規範」。

考場最常見的四大 `ValueError` 引爆場景：
1. **小數字串直接丟給 `int()` 轉型失敗**：
   若測資輸入為浮點數字串（如 `"3.14"`），直接執行 `int("3.14")` 會當場引爆 `ValueError: invalid literal for int() with base 10: '3.14'`！
   - *正解*：若可能包含小數點，必須先轉浮點數再轉整數：`int(float("3.14"))`，或是字串先依小數點切開取前半段。
2. **空字串或純空白行轉整數**：
   在讀取未知測資時，若讀到了空行 `""`，直接執行 `int("")` 會立刻引爆 `ValueError`。
3. **多變數解包數量不對稱（Unpacking Mismatch）**：
   例如寫 `a, b = map(int, input().split())`，但官方該筆測資只給了 1 個整數（噴出 `not enough values to unpack`）或給了 3 個整數（噴出 `too many values to unpack`）。
4. **`list.index(x)` 查無元素時的暴斃特性**：
   初學者常以為 `arr.index(target)` 在找不到時會像其他語言一樣回傳 `-1`，但 Python 會毫不留情地直接拋出 `ValueError: 'x' is not in list` 讓整份程式當場崩潰！


In [ ]:
# 13.3.3 程式碼演示：四類常見 ValueError 與先驗防禦方案
print("--- 場景 1: 小數字串轉整數失敗 vs 兩段式轉型 ---")
float_str = "42.8"
try:
    bad_int = int(float_str)
except ValueError as e:
    print(f"  [捕捉錯誤 1] int('{float_str}') 失敗: {e}")

# 正確防禦：兩段式轉換或字串截斷
safe_int = int(float(float_str))
print(f"  [安全轉型] int(float('{float_str}')) = {safe_int}")

print("\n--- 場景 2: 輸入行解包數量不符防禦 ---")
raw_input_line = "100 200 300" # 輸入了 3 個數字
try:
    a, b = map(int, raw_input_line.split()) # 只宣告 2 個變數，解包數量不符！
except ValueError as e:
    print(f"  [捕捉錯誤 2] 解包失敗: {e}")

# 正確防禦：先用串列接收，再檢查長度
tokens = list(map(int, raw_input_line.split()))
if len(tokens) >= 2:
    a, b = tokens[0], tokens[1]
    print(f"  [安全解包] 安全取出前兩項: a={a}, b={b}, 忽略多餘的項")

print("\n--- 場景 3: list.index() 找不到元素之暴斃防禦 ---")
colors = ["red", "green", "blue"]
target_color = "yellow"

try:
    idx = colors.index(target_color)
except ValueError as e:
    print(f"  [捕捉錯誤 3] 查找失敗: {e}")

# 正確防禦：永遠先以 in 關鍵字做成員檢查
if target_color in colors:
    idx = colors.index(target_color)
else:
    idx = -1 # 依照競賽慣例安全回傳 -1
print(f"  [安全查找] 搜尋 {target_color} 結果: 索引 = {idx}")


### 13.3.3 語法重點回顧與核心觀念提煉

防禦 `ValueError` 的四大必備防線：
1. **數值字串轉型兩原則**：
   - 純整數用 `int(s)`。
   - 帶小數點字串轉整數：`int(float(s))`。
2. **搜尋前必備守衛**：在呼叫 `arr.index(target)` 之前，腦中必須自動補上 `if target in arr:`。
3. **動態輸入安全接水桶**：只要輸入行的數量不可控，一律寫 `arr = list(map(int, input().split()))`，再用 `len(arr)` 進行防守，絕不貿然解包！


In [ ]:
# 13.3.3 學生實作練習：彈性整數剖析與索引搜尋器
# 任務說明：實作 robust_parse_and_find(text_list, target_val) 函式
# 給定一個包含可能格式不一的字串串列 text_list（如 ["10", "20.5", "abc", "30"]）
# 需求：
# 1. 逐項嘗試將其轉換為整數（若為 "20.5" 請安全截斷為 20；若為 "abc" 則略過）
# 2. 將所有成功轉換出的整數依序放入有效串列 valid_numbers 中
# 3. 在 valid_numbers 中尋找 target_val，若找到回傳其索引；若查無此數回傳 -1（不可引發 ValueError！）

def robust_parse_and_find(text_list: list, target_val: int) -> int:
    valid_numbers = []
    # 1. 安全剖析
    for s in text_list:
        try:
            val = int(float(s))
            valid_numbers.append(val)
        except ValueError:
            pass
            
    # 2. 安全查找
    if target_val in valid_numbers:
        return valid_numbers.index(target_val)
    return -1

# 測試用例
sample_data = ["10", "3.14", "hello", "50", "99.9"]
print("尋找 3 的位置 (3.14 轉為 3):", robust_parse_and_find(sample_data, 3))
print("尋找 99 的位置 (99.9 轉為 99):", robust_parse_and_find(sample_data, 99))
print("尋找不存在的 100:", robust_parse_and_find(sample_data, 100))


In [ ]:
# 13.3.3 單元測試驗證
data = ["10", "20", "invalid", "30.5"]
assert robust_parse_and_find(data, 10) == 0
assert robust_parse_and_find(data, 30) == 2
assert robust_parse_and_find(data, 999) == -1
assert robust_parse_and_find([], 5) == -1
print("🎉 13.3.3 所有測試通過！徹底掃除 ValueError 轉型與搜尋地雷！")


### 13.3.4 字典與鍵值遺漏：KeyError 與 set.remove 陷阱

字典（`dict`）與集合（`set`）是 APCS 實作題中實現 $O(1)$ 極速頻率統計與去重查表的最強神兵。然而，如果存取了不存在的鍵，Python 會立刻引爆 `KeyError`！

考場最常見的三大 `KeyError` 引爆死角：
1. **頻率統計計數器未初始化**：
   在統計字母或單字頻率時，初學者常直接寫 `counts[ch] += 1`。當第一次遇到某個字元時，字典中尚未建立此鍵，立刻引發崩潰！
2. **對照表缺少預設邊界**：
   建立符號轉代碼字典時，測資輸入了未知字元，使用 `mapping[x]` 直接暴斃。
3. **集合 `set.remove()` 找不到元素的隱形炸彈**：
   在維護集合元素時，許多初學者使用 `my_set.remove(x)`。若 `x` 不在集合中，Python 會當場拋出 `KeyError: x`！
   - *神級防禦*：改用 **`my_set.discard(x)`**！當元素存在時它會將其刪除；當元素不存在時，它會「平靜地什麼都不做」，完全不會引爆任何例外！

針對字典，核心防禦神技永遠是：**`dict.get(key, default)`**！


In [ ]:
# 13.3.4 程式碼演示：KeyError 現場與 dict.get()、set.discard() 安全神技
print("--- 場景 1: 字典計數器未初始化崩潰 vs .get() 一行搞定 ---")
text = "banana"
bad_counts = {}

try:
    for ch in text:
        # 第一次遇到 'b' 時，字典是空的，bad_counts['b'] 引爆 KeyError！
        bad_counts[ch] += 1
except KeyError as e:
    print(f"  [捕捉錯誤 1] KeyError: 字典中查無鍵值 {e}")

# 考場必背極速防禦寫法：
good_counts = {}
for ch in text:
    good_counts[ch] = good_counts.get(ch, 0) + 1
print(f"  [安全計數結果] {good_counts}")

print("\n--- 場景 2: set.remove() 崩潰 vs set.discard() 安全除法 ---")
active_users = {"Alice", "Bob"}
target_to_remove = "Charlie" # 不在集合中

try:
    active_users.remove(target_to_remove)
except KeyError as e:
    print(f"  [捕捉錯誤 2] set.remove({e}) 拋出 KeyError！")

# 安全防禦方案：改用 discard()
active_users.discard(target_to_remove) # 安全略過，完全不崩潰！
print(f"  [安全刪除驗證] 使用 discard 後集合狀態依然完好: {active_users}")

print("\n--- 場景 3: 二維座標拜訪對照表安全存取 ---")
visited_dist = {(0, 0): 0, (0, 1): 1}
query_coord = (5, 5) # 未曾造訪之座標

# 使用 .get() 附帶預設值 -1 表示尚未造訪
dist = visited_dist.get(query_coord, -1)
print(f"  查詢座標 {query_coord} 距離: {dist} (安全回傳 -1)")


### 13.3.4 語法重點回顧與核心觀念提煉

防禦 `KeyError` 的三大考場自律紀律：
1. **字典計數永遠寫 `d[k] = d.get(k, 0) + 1`**：把這行代碼刻在骨子裡，考場寫頻率統計 100% 免疫 KeyError。
2. **集合刪除永遠寫 `s.discard(x)`**：除非題目明確要求「找不到時必須報錯」，否則在競賽中一律使用 `discard()` 取代 `remove()`。
3. **安全預設查表**：查詢字典對照表時，一律寫 `val = mapping.get(key, "UNKNOWN")`，絕不直接使用中括號裸查。


In [ ]:
# 13.3.4 學生實作練習：安全購物車結帳統計器
# 任務說明：實作 calculate_cart_total(price_table, cart_items) 函式
# 傳入商品單價字典 price_table（如 {"apple": 20, "banana": 15}）
# 傳入顧客購物車清單 cart_items（如 ["apple", "apple", "orange", "banana"]）
# 需求：
# 1. 遍歷購物車清單計算總金額
# 2. 若購物車中出現 price_table 查不到的未知商品（如 "orange"），
#    請使用 .get() 自動將其單價視為 0 元，絕不可引發 KeyError 崩潰！
# 回傳購物車總花費！

def calculate_cart_total(price_table: dict, cart_items: list) -> int:
    total = 0
    # 請在此處使用 .get() 累加總額
    for item in cart_items:
        total += price_table.get(item, 0)
    return total

# 測試用例
prices = {"apple": 30, "banana": 20, "milk": 50}
items = ["apple", "banana", "apple", "chocolate", "milk"] # chocolate 不在目錄中
print("購物車總花費:", calculate_cart_total(prices, items))


In [ ]:
# 13.3.4 單元測試驗證
menu = {"tea": 25, "coffee": 55}
assert calculate_cart_total(menu, ["tea", "tea"]) == 50
assert calculate_cart_total(menu, ["coffee", "water"]) == 55 # water 查無，計 0 元
assert calculate_cart_total(menu, []) == 0
assert calculate_cart_total({}, ["anything"]) == 0
print("🎉 13.3.4 所有測試通過！徹底征服 KeyError 與集合刪除陷阱！")


### 13.3.5 算術與型態衝突：ZeroDivisionError 與 TypeError

在處理數學計算與變數運算時，有兩種不可忽視的例外常導致考場慘劇：**`ZeroDivisionError`** 與 **`TypeError`**。

#### 1. 除以零與模除零：`ZeroDivisionError`
在數學上，除以零是未定義的。在 Python 中：
- `/`、`//`、`%` 只要右邊的除數是 `0`，直譯器就會當場暴斃！
- 考場極端測資最常出現在：計算平均值 `total / count` 時，剛好整組測資沒有任何項目符合條件導致 `count == 0`；或是幾何題目兩點 $x$ 座標相同時計算垂直線斜率。

#### 2. 型態操作衝突：`TypeError`
Python 是「強型別（Strongly Typed）」語言，不同型態之間絕不會暗中隨便相容。
考場最高頻的三大 `TypeError` 翻車現場：
- **字串與數字以 `+` 串接**：`"排名: " + rank` $\to$ 引爆 `can only concatenate str to str`。
- **誤將容器括號打成小括號呼叫**：`arr = [1, 2, 3]; val = arr(0)` $\to$ 引爆 `'list' object is not callable`（初學者常手滑把中括號打成圓括號）。
- **對無長度物件取 `len()`**：`len(12345)` $\to$ 引爆 `object of type 'int' has no len()`。


In [ ]:
# 13.3.5 程式碼演示：ZeroDivisionError 與三類常見 TypeError 排查
print("--- 案例 1: 模除為零 ZeroDivisionError ---")
divisor = 0
try:
    ans = 100 % divisor
except ZeroDivisionError as e:
    print(f"  [捕捉錯誤 1] ZeroDivisionError: {e}")
    print("  說明：不論是 // 還是 %，右側除數為 0 一律當場暴斃！")

print("\n--- 案例 2: 字串串接型態衝突 vs f-string 自動轉型 ---")
score = 95
try:
    bad_msg = "得分是: " + score # 試圖用加號串接整數
except TypeError as e:
    print(f"  [捕捉錯誤 2] TypeError: {e}")

# 正確防禦：使用 f-string，內部自動呼叫 str()，絕不引發 TypeError
good_msg = f"得分是: {score}"
print(f"  [安全格式化] {good_msg}")

print("\n--- 案例 3: 誤將串列當中括號函式呼叫 'list' object is not callable ---")
scores = [80, 90, 100]
try:
    val = scores(0) # 手滑把 scores[0] 打成圓括號 scores(0)
except TypeError as e:
    print(f"  [捕捉錯誤 3] TypeError: {e}")
    print("  💡 診斷提示：看到 'not callable'，立刻檢查代碼中是不是把串列的中括號打成小括號了！")


### 13.3.5 語法重點回顧與核心觀念提煉

算術與型態衝突的兩大神盾：
1. **分母必加守衛條件**：
   ```python
   avg = total / count if count != 0 else 0.0
   ```
2. **終結 `+` 號字串拼接，全面改用 `f-string`**：
   `f"{prefix} {number}"` 不僅執行速度更快，還能自動將所有物件安全轉為字串，從根本上徹底根除 `TypeError`。
3. **小括號 vs 中括號肌肉記憶**：
   - 存取容器元素：永遠使用**中括號 `arr[i]`、`dict[key]`**。
   - 呼叫函式：才使用**小括號 `func(arg)`**。


In [ ]:
# 13.3.5 學生實作練習：安全平均值與標籤格式化
# 任務說明：實作 format_average_label(numbers, label) 函式
# 傳入整數串列 numbers 與前綴標籤字串 label（如 label="成績"）
# 需求：
# 1. 若 numbers 為空串列，平均值視為 0.0（防範 ZeroDivisionError）
# 2. 計算 numbers 的浮點數平均值 avg
# 3. 使用 f-string 組合字串回傳：f"{label}: {avg:.1f}"（防範 TypeError）

def format_average_label(numbers: list, label: str) -> str:
    # 請在此處實作分母防禦與 f-string 格式化
    if not numbers:
        avg = 0.0
    else:
        avg = sum(numbers) / len(numbers)
    return f"{label}: {avg:.1f}"

# 測試用例
print("正常計算:", format_average_label([80, 90, 100], "班級平均"))
print("空串列防禦:", format_average_label([], "缺考平均"))


In [ ]:
# 13.3.5 單元測試驗證
assert format_average_label([10, 20, 30], "總分") == "總分: 20.0"
assert format_average_label([], "測試") == "測試: 0.0"
assert format_average_label([5], "單項") == "單項: 5.0"
print("🎉 13.3.5 所有測試通過！徹底征服除以零與各類 TypeError 陷阱！")


### 13.3.6 邊界防禦第一原則：在存取前以條件式築起護城河（LBYL）

在軟體工程與競賽程式設計中，有一種廣為人知的防禦哲學稱為**「Look Before You Leap（LBYL，三思而後行）」**。它的核心思維非常純粹：在執行任何可能導致災難的危險動作之前，先用嚴密的 `if` 條件判斷做前置檢查；只有當一切條件完全合法時，才允許程式跨出那一步。

在 APCS 考場上，LBYL 是對抗 Runtime Error 最直接、最迅速且最容易排查的武器。相較於事後收拾殘局，事先預防具有以下三大壓倒性優勢：
1. **防止非預期狀態擴散**：若不在源頭攔截，錯誤值可能會流入後續複雜的演算法管線中，最終演變成極難定位的邏輯錯誤（WA）。
2. **保護演算法極端邊界**：許多圖論或動態規劃題目，最常在 $N=0$ 或 $N=1$ 的微型測資上翻車。只要在主函式入口處加上 2 行邊界條件特殊處理（Edge Case Guard），就能穩穩守住關鍵分數。
3. **保持程式流程清晰單純**：利用守衛條件（Guard Clauses），不合法的輸入在最前線就直接被過濾或回傳，主幹程式碼便可以放心地在理想的假設下專注執行核心演算法。

隨時牢記邊界防禦的口訣：「查索引先看長度、做除法先看分母、查字典先用 get、做字串先看空值」。


In [ ]:
# 13.3.6 程式碼演示：考場真實模擬 —— 機器人地圖走訪之全方位 LBYL 護城河
# 題目情境：機器人根據指令在 H x W 的網格地圖中移動，途中需要消耗能量並計算平均效率
# 潛伏危險：出界（IndexError）、無效指令（KeyError）、步數為零求平均（ZeroDivisionError）

def simulate_robot_trip(H, W, start_r, start_c, commands, energy_grid):
    print(f"=== 機器人啟航：地圖 {H}x{W}，起點 ({start_r}, {start_c}) ===")
    r, c = start_r, start_c
    total_energy = 0
    valid_steps = 0
    
    move_map = {
        'U': (-1, 0),
        'D': (1, 0),
        'L': (0, -1),
        'R': (0, 1)
    }
    
    for cmd in commands:
        # LBYL 防衛 1: 檢查指令是否合法（防 KeyError）
        if cmd not in move_map:
            print(f"  ⚠️ 忽略無效指令 '{cmd}'")
            continue
            
        dr, dc = move_map[cmd]
        nr, nc = r + dr, c + dc
        
        # LBYL 防衛 2: 檢查移動後座標是否出界（防 IndexError & 負數幽靈）
        if not (0 <= nr < H and 0 <= nc < W):
            print(f"  ❌ 撞牆防禦：指令 '{cmd}' 會導致移動至出界座標 ({nr}, {nc})，原地待命！")
            continue
            
        # 安全放行移動
        r, c = nr, nc
        total_energy += energy_grid[r][c]
        valid_steps += 1
        print(f"  ✅ 成功移動至 ({r}, {c})，吸收能量 {energy_grid[r][c]}")
        
    # LBYL 防衛 3: 計算平均每步能量消耗（防 ZeroDivisionError）
    if valid_steps > 0:
        avg_energy = total_energy / valid_steps
    else:
        avg_energy = 0.0 # 一步都沒走成時的安全預設值
        
    print(f"🏁 終點: ({r}, {c}) | 總步數: {valid_steps} | 平均能量: {avg_energy:.2f}\n")
    return (r, c, avg_energy)

# 測試地圖 2x2
sample_grid = [
    [10, 20],
    [30, 40]
]
# 測試指令集包含：合法移動、撞牆出界、未知指令
simulate_robot_trip(2, 2, 0, 0, ['R', 'R', 'D', 'X', 'L'], sample_grid)


### 13.3.6 語法重點回顧與核心觀念提煉

全方位 LBYL 護城河架構心法：
1. **輸入與指令防禦**：進入運算前先用 `in` 驗證合法性。
2. **座標與索引防禦**：進入容器前先用 `0 <= idx < bound` 確立安全區間。
3. **分母除法防禦**：執行除號前必設 `if count > 0:`。
只要在程式關鍵路口把守這三道關卡，你的程式碼在 APCS 考場上就宛如穿上了重裝防彈衣，無論後端丟出多麼刁鑽的極端測資，都能屹立不倒！


In [ ]:
# 13.3.6 學生實作練習：安全陣列中位數與全域極值聚合器
# 任務說明：實作 robust_stats_summary(arr) 函式
# 給定一個整數串列 arr，計算並回傳包含 "min", "max", "median" 的字典
# 嚴格 LBYL 防禦要求：
# 1. 若傳入空串列 []，不可崩潰（防 IndexError），應安全回傳 None！
# 2. 計算中位數時，若長度為奇數取正中央數值，偶數取中央兩數之平均（float）
# 3. 排序運算不得破壞傳入的原串列 arr（防副作用）！

def robust_stats_summary(arr: list):
    # LBYL 防禦 1: 空串列檢查
    if not arr:
        return None
        
    # 防禦性排序（不改動原串列）
    sorted_data = sorted(arr)
    n = len(sorted_data)
    
    min_val = sorted_data[0]
    max_val = sorted_data[-1]
    
    mid_idx = n // 2
    if n % 2 == 1:
        median_val = float(sorted_data[mid_idx])
    else:
        median_val = (sorted_data[mid_idx - 1] + sorted_data[mid_idx]) / 2.0
        
    return {
        "min": min_val,
        "max": max_val,
        "median": median_val
    }

# 測試用例
print("奇數長度統計:", robust_stats_summary([5, 1, 9, 3, 7]))
print("偶數長度統計:", robust_stats_summary([10, 20, 30, 40]))
print("空串列邊界防禦:", robust_stats_summary([]))


In [ ]:
# 13.3.6 單元測試驗證
assert robust_stats_summary([]) is None
res1 = robust_stats_summary([3, 1, 2])
assert res1["min"] == 1 and res1["max"] == 3 and res1["median"] == 2.0
res2 = robust_stats_summary([10, 20, 30, 40])
assert res2["median"] == 25.0
# 驗證原串列未被竄改
original = [5, 2, 8]
robust_stats_summary(original)
assert original == [5, 2, 8], "原串列不可被副作用竄改！"
print("🎉 13.3.6 所有測試通過！恭喜你已修成百毒不侵的 RE 邊界防禦大師！")


## 13.3 總結與考場除錯防禦全景對照表

在本單元中，我們系統性地拆解了直譯器執行時期崩潰（Runtime Error, RE）的發生機制與 APCS 考場最常見的五大例外殺手。牢記以下各類例外的對應防禦手段，是你攀登 APCS 實作滿分的核心防彈衣：

| 例外名稱（Exception） | 觸發根本成因 | 典型災難場景 | 最佳防禦黃金法則 |
| :--- | :--- | :--- | :--- |
| **`IndexError`** | 存取超出範圍之索引 | 迴圈查相鄰 `arr[i+1]`、空串列取 `arr[0]`、網格負數穿牆 | 迴圈縮減上界 `len-1`，取值前 `if 0 <= i < len:`，網格雙向檢查 |
| **`ValueError`** | 傳入數值內容不合規範 | `int("3.14")` 轉型失敗、解包個數不符、`arr.index(x)` 查無元素 | 浮點字串兩段式轉型、解包前先用 list 接收、搜尋前先 `if x in arr:` |
| **`KeyError`** | 字典中查無指定之鍵 | 頻率統計未初始化、`set.remove(x)` 元素不存在 | 存取全面改用 `dict.get(key, default)`，集合刪除改用 `set.discard(x)` |
| **`ZeroDivisionError`** | 除數或模除運算元為零 | 計算平均時計數為 0、計算斜率兩點垂直 | 除法運算前必設守衛：`if denominator != 0:` |
| **`TypeError`** | 型態不支援該運算子 | 字串以 `+` 串接整數、誤將串列當中括號呼叫 `arr(0)` | 字串拼接全面升級為 `f"{var}"` 格式化字串，認清中括號與圓括號 |

### 🚀 下一步學習指引
雖然運用「三思而後行（LBYL）」的 `if` 條件式能擋下絕大多數已知風險，但在真實的競賽環境中，有些狀況難以透過簡單的 `if` 預判——最著名的例子就是 APCS 題目常見的**「未知行數輸入（讀取至檔案結尾 EOF）」**。
在下一單元 **13-4《例外捕捉語法：try ... except 架構與未知長度輸入處理》** 中，我們將正式學習 Python 原生的主動防禦框架 `try ... except`，並掌握考場必備的 `EOFError` 終端讀檔神技！
